# Bu notebookda projede kullanılacak makine öğrenim süreçleri modeller ve searchler bulunmaktadır


vektörize edilmiş market databaseimiz için search meselesi


In [ ]:
import joblib
import polars as pl
from rapidfuzz import fuzz
from sklearn.metrics.pairwise import cosine_similarity


loaded_vectorizer = joblib.load('tfidf_vectorizer.pkl')
loaded_matrix = joblib.load('tfidf_matrix.pkl')
intent_clf = joblib.load('intent_classifier.pkl')
loaded_df = pl.read_parquet('cleaned_dataframe.parquet')



def metin_on_isleme(metin):
    harfler = {"ç": "c", "ş": "s", "ğ": "g", "ı": "i", "ö": "o", "ü": "u", "Ç": "c", "Ş": "s", "Ğ": "g", "İ": "i", "Ö": "o", "Ü": "u"}
    for tr, eng in harfler.items():
        metin = metin.replace(tr, eng)
    return metin.lower()


def kategori_temizle(metin):
    harfler = {"ç": "c", "ş": "s", "ğ": "g", "ı": "i", "ö": "o", "ü": "u", "Ç": "c", "Ş": "s", "Ğ": "g", "İ": "i", "Ö": "o", "Ü": "u"}
    for tr, eng in harfler.items():
        metin = metin.replace(tr, eng)
    return metin.lower().replace("-", " ").replace("/", " ")

def organik_arama_motoru(malzeme_adi, top_n=3):
    temiz_malzeme = metin_on_isleme(malzeme_adi)
    birlesik_malzeme = temiz_malzeme.replace(" ", "")
    
    
    vec_ayri = loaded_vectorizer.transform([temiz_malzeme])
    vec_birlesik = loaded_vectorizer.transform([birlesik_malzeme])
    
    skor_ayri = cosine_similarity(vec_ayri, loaded_matrix).flatten()
    skor_birlesik = cosine_similarity(vec_birlesik, loaded_matrix).flatten()
    
    if skor_birlesik.max() > skor_ayri.max():
        aktif_skorlar = skor_birlesik
        aktif_kelime = birlesik_malzeme
    else:
        aktif_skorlar = skor_ayri
        aktif_kelime = temiz_malzeme
        
  
    kelimeler = temiz_malzeme.split()
    ana_isim = kelimeler[-1] if kelimeler else ""
    
    if len(kelimeler) > 1:
        niyet_metni = f"{ana_isim} {ana_isim}"
    else:
        niyet_metni = temiz_malzeme
        
    niyet_vektoru = loaded_vectorizer.transform([niyet_metni])
    kategori_olasiliklari = intent_clf.predict_proba(niyet_vektoru)[0]
    siniflar = intent_clf.classes_
    niyet_sozlugu = dict(zip(siniflar, kategori_olasiliklari))
    
    tahmin_edilen_kategori = siniflar[kategori_olasiliklari.argmax()]
    print(f"\n--- Girdi: '{malzeme_adi}' | AI Niyet: %{kategori_olasiliklari.max()*100:.0f} '{tahmin_edilen_kategori}' ---")
    
    aday_indeksler = aktif_skorlar.argsort()[-20:][::-1]
    aday_listesi = []
    
    for idx in aday_indeksler:
        tfidf_skor = aktif_skorlar[idx]
        urun = loaded_df["ITEMNAME"][int(idx)]
        kat1 = loaded_df["CATEGORY1"][int(idx)]
        kat2 = loaded_df["CATEGORY2"][int(idx)]
        
        urun_temiz = metin_on_isleme(urun)
        fuzzy_skor = fuzz.token_set_ratio(aktif_kelime, urun_temiz) / 100.0
        niyet_skoru = niyet_sozlugu.get(kat1, 0.0)
        
        
        kategori_metni = kategori_temizle(f"{kat1} {kat2}")
        kategori_kelimeleri = kategori_metni.split()
        
        kategori_bonusu = 0.0
        if temiz_malzeme in kategori_metni:
            kategori_bonusu = 0.50
        elif ana_isim in kategori_kelimeleri:
            kategori_bonusu = 0.30
            
        final_skor = (tfidf_skor * 0.30) + (fuzzy_skor * 0.40) + (niyet_skoru * 0.10) + kategori_bonusu
        
        aday_listesi.append({
            "urun": urun, "kat1": kat1, "kat2": kat2, 
            "skor": final_skor, 
            "detay": f"(TF-IDF: {tfidf_skor:.2f} | Fuzzy: {fuzzy_skor:.2f} | Kat.Bonus: {kategori_bonusu:.2f})"
        })
        
    aday_listesi = sorted(aday_listesi, key=lambda x: x["skor"], reverse=True)
    
    for i in range(min(top_n, len(aday_listesi))):
        res = aday_listesi[i]
        print(f"[{res['skor']:.2f}] [{res['kat1']} > {res['kat2']}] {res['urun']} {res['detay']}")


organik_arama_motoru("sıvı yağ")
organik_arama_motoru("kara biber")
organik_arama_motoru("biber salçası")
organik_arama_motoru("150 g rendelenmiş kaşar peyniri")


Modeller yükleniyor...
Sistem Hazır!

--- Girdi: 'sıvı yağ' | AI Niyet: %37 'KOZMETIK' ---
[0.82] [GIDA > SIVI YAG] ORUCOGLU IDEAL YAG 5 LT PET*4* (TF-IDF: 0.30 | Fuzzy: 0.55 | Kat.Bonus: 0.50)
[0.82] [GIDA > SIVI YAG] ORUCOGLU IDEAL YAG 5 LT TNK*4* (TF-IDF: 0.30 | Fuzzy: 0.55 | Kat.Bonus: 0.50)
[0.39] [KOZMETIK > DUS-BANYO] S DURU SIVI SABUN 500 ML (TF-IDF: 0.30 | Fuzzy: 0.67 | Kat.Bonus: 0.00)

--- Girdi: 'kara biber' | AI Niyet: %75 'GIDA' ---
[0.67] [GIDA > TUZ-BAHARAT] ZENGIBAR KARABIBER 100 GR (TF-IDF: 0.65 | Fuzzy: 1.00 | Kat.Bonus: 0.00)
[0.66] [GIDA > TUZ-BAHARAT] NERGIS PET KARABIBER 130 GR.*20* (TF-IDF: 0.62 | Fuzzy: 1.00 | Kat.Bonus: 0.00)
[0.66] [GIDA > TUZ-BAHARAT] DOKME KARABIBER 10 KG.  (TF-IDF: 0.62 | Fuzzy: 1.00 | Kat.Bonus: 0.00)

--- Girdi: 'biber salçası' | AI Niyet: %29 'GIDA' ---
[0.39] [GIDA > HAZIR YEMEK-KONSERVE-SALCA] TAT DOMATES SALCASI 1/5  *48* (TF-IDF: 0.28 | Fuzzy: 0.70 | Kat.Bonus: 0.00)
[0.33] [GIDA > HAZIR YEMEK-KONSERVE-SALCA] OLCA BIB.SALCASI 4200 G

kısaca yukarda ne yaptığımı anlatayım bizim yemek tariflerinden çıkaracağımız malzemeleri market database'imizde bulmamız lazımdı bunun için search algoritma modellerine ihtiyacımız vardı ilk adım olarak tf-idf ile kelimelerimizin matematiğini çıkardık sonrasında kosinüs benzerliği ile basit bi algoritma kurdum ancak domates - domates salçası aratmasında aynı sonuçlar geldi çünkü ikisi de kosinüs olarak neredeyse aynı ama farklı şeyler bunun sonucunda oyuna rapidfuzz geldi ve cümle uzunluklarını kattı sonra salça aratması yaptım alakasız şeyler geldi çünkü türkçe karakterleri anlamıyordu algoritma basit bi türkçe karakter dönüşümü ekledim bu sorun da böyle çözüldü sonrasında kara biber ve karabiber araması yaptım apayrı sonuçlar çıktı sonrasında boşluk için birleşik arama da ekledim algoritma güzel çalıştı şuana kadar ama sıvı yağ dediğimiz zaman patladı çünkü sıvı çok fazla yerde geçiyor o yüzden niyet tahmini ekliyoruz bayes gelicek buraya niyet odaklıyı ekledim ama yine patladı çünkü sıvı kelimesi kozmetikde daha çok geçiyo benim bu niyetin içine dil yapısını ekleyip sıvının bir sıfat olduğunu öğretmem lazım (bu fikrime Part of Speech (POS) Tagging" (Kelime Türü İşaretleme) ve "Head-Directionality" (Merkez Yönlülük) deniyormuş bu arada yeni öğrendim :)  bunlar da bizi sıvı yağa götürmedi en başta yapmam gereken meseleyi şimdi ekleyip kategori eşleşmesi ekledim